# Тема 7. Обучение без учителя: PCA и кластеризация

Все предыдущие темы были посвящены **обучению с учителем**: у каждого объекта есть правильный ответ, и задача — научиться его предсказывать.

Теперь переходим к другому классу задач — **обучению без учителя**. Меток нет. Задача — найти структуру в самих данных: сжать их, визуализировать, сгруппировать похожие объекты.

Два главных инструмента на этом занятии:
- **PCA (метод главных компонент)** — снижение размерности.
- **Кластеризация** — автоматическая группировка объектов.

### Содержание
1. [Метод главных компонент (PCA)](#1.-Метод-главных-компонент-(PCA))
2. [PCA как предобработка](#2.-PCA-как-предобработка)
3. [Кластеризация: k-Means](#3.-Кластеризация:-k-Means)
4. [Выбор числа кластеров](#4.-Выбор-числа-кластеров)
5. [Другие алгоритмы кластеризации](#5.-Другие-алгоритмы-кластеризации)
6. [Метрики качества кластеризации](#6.-Метрики-качества-кластеризации)
7. [Практика](#7.-Практика)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import seaborn as sns
sns.set()
from matplotlib import pyplot as plt
from scipy.spatial.distance import cdist, pdist
from scipy.cluster import hierarchy

from sklearn import datasets, metrics
from sklearn.cluster import (
    AffinityPropagation, AgglomerativeClustering, KMeans, SpectralClustering
)
from sklearn.datasets import load_digits, load_iris
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.manifold import TSNE
from sklearn.metrics import (
    accuracy_score, adjusted_rand_score, classification_report,
    homogeneity_score, completeness_score, silhouette_score, v_measure_score
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

%config InlineBackend.figure_format = 'svg'

---
## 1. Метод главных компонент (PCA)

### Идея

Представьте, что у вас 100 признаков. Многие из них коррелируют друг с другом — несут похожую информацию. Можно ли описать данные с помощью меньшего числа «сводных» признаков без большой потери информации?

PCA отвечает на этот вопрос: он находит новые оси (главные компоненты), вдоль которых данные **варьируются максимально**. Первая компонента объясняет наибольшую долю дисперсии, вторая — следующую по величине, и так далее.

### Математика: SVD

PCA использует **сингулярное разложение (SVD)** матрицы данных:

$$X = U D V^T$$

где $U \in \mathbb{R}^{m \times m}$, $V \in \mathbb{R}^{n \times n}$ — ортогональные матрицы, $D \in \mathbb{R}^{m \times n}$ — диагональная матрица сингулярных значений.

Главные компоненты — это столбцы матрицы $V$. Проекция данных на первые $k$ компонент:
$$Z = X V_k \in \mathbb{R}^{m \times k}$$

**Алгоритм PCA:**
1. Центрировать данные: $x_i \leftarrow x_i - \bar{x}$
2. При необходимости — нормировать по std
3. Вычислить SVD матрицы $X$
4. Взять первые $k$ правых сингулярных векторов
5. Спроецировать данные: $Z = X V_k$

In [ ]:
# Простейший пример на 2D данных
np.random.seed(0)
mean = np.array([0.0, 0.0])
cov  = np.array([[1.0, -1.0], [-2.0, 3.0]])
X_2d = np.random.multivariate_normal(mean, cov, 300)

pca_2d = PCA()
pca_2d.fit(X_2d)

print('Доля объяснённой дисперсии:')
for i, ratio in enumerate(pca_2d.explained_variance_ratio_):
    print(f'  {i+1}-я компонента: {ratio*100:.1f}%')
print('\nНаправления главных компонент:')
for i, comp in enumerate(pca_2d.components_):
    print(f'  PC{i+1}: {comp}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Исходные данные
axes[0].scatter(X_2d[:, 0], X_2d[:, 1], alpha=0.4, s=20, color='steelblue')
origin = pca_2d.mean_
for i, (comp, var) in enumerate(zip(pca_2d.components_, pca_2d.explained_variance_)):
    axes[0].annotate('', xy=origin + np.sqrt(var) * comp,
                     xytext=origin,
                     arrowprops=dict(arrowstyle='->', color=['orange','red'][i], lw=2))
    axes[0].text(*(origin + np.sqrt(var)*comp*1.1), f'PC{i+1}', fontsize=11,
                 color=['orange','red'][i])
axes[0].set_title('Исходные данные и главные компоненты')
axes[0].set_aspect('equal')
axes[0].grid(True, alpha=0.3)

# Проекция на 1-ю компоненту
pca_1 = PCA(0.90)
X_reduced = pca_1.fit_transform(X_2d)
X_back = pca_1.inverse_transform(X_reduced)
axes[1].scatter(X_2d[:, 0], X_2d[:, 1], alpha=0.2, s=15, color='steelblue', label='Исходные')
axes[1].scatter(X_back[:, 0], X_back[:, 1], alpha=0.6, s=15, color='orange', label='Проекция (90% дисп.)')
axes[1].set_title(f'Проекция на 1-ю компоненту ({pca_1.n_components_} из 2)')
axes[1].legend()
axes[1].set_aspect('equal')
axes[1].grid(True, alpha=0.3)
plt.tight_layout()

### PCA на датасете Iris: визуализация 4D → 2D

In [ ]:
iris = load_iris()
X_ir, y_ir = iris.data, iris.target

pca_ir = PCA(n_components=2)
X_ir_pca = pca_ir.fit_transform(X_ir)

print('Интерпретация компонент:')
for i, comp in enumerate(pca_ir.components_):
    expr = ' + '.join(f'{v:.3f}×{n}' for v, n in zip(comp, iris.feature_names))
    print(f'  PC{i+1} ({pca_ir.explained_variance_ratio_[i]*100:.1f}%): {expr}')

plt.figure(figsize=(7, 5))
colors = ['steelblue', 'orange', 'green']
for cls, name, color in zip([0,1,2], iris.target_names, colors):
    mask = y_ir == cls
    plt.scatter(X_ir_pca[mask, 0], X_ir_pca[mask, 1],
                label=name, color=color, s=40, edgecolors='white', alpha=0.8)
plt.xlabel(f'PC1 ({pca_ir.explained_variance_ratio_[0]*100:.1f}%)')
plt.ylabel(f'PC2 ({pca_ir.explained_variance_ratio_[1]*100:.1f}%)')
plt.title('Iris: PCA-проекция 4D → 2D')
plt.legend()
plt.grid(True, alpha=0.3)

Два первых компонента сохраняют более 97% дисперсии. Классы в 2D пространстве видны гораздо отчётливее, чем при случайном выборе двух исходных признаков.

### PCA на рукописных цифрах: 64D → 2D

In [ ]:
digits = load_digits()
X_dig, y_dig = digits.data, digits.target

# Покажем несколько цифр
plt.figure(figsize=(12, 3))
for i in range(10):
    plt.subplot(1, 10, i+1)
    plt.imshow(X_dig[i].reshape(8, 8), cmap='gray')
    plt.title(str(y_dig[i]))
    plt.axis('off')
plt.suptitle('Примеры рукописных цифр (8×8 пикселей = 64 признака)', y=1.02)

In [ ]:
pca_dig = PCA(n_components=2)
X_dig_pca = pca_dig.fit_transform(X_dig)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sc = axes[0].scatter(X_dig_pca[:, 0], X_dig_pca[:, 1],
                     c=y_dig, edgecolor='none', alpha=0.7, s=20,
                     cmap=plt.cm.get_cmap('tab10', 10))
plt.colorbar(sc, ax=axes[0])
axes[0].set_title('PCA: 64D → 2D')
axes[0].set_xlabel('PC1')
axes[0].set_ylabel('PC2')
axes[0].grid(True, alpha=0.3)

# Первые две главные компоненты как изображения
for i, (ax, title) in enumerate([
    (axes[1], 'Первые 2 главные компоненты')
]):
    fig2, ax2s = plt.subplots(1, 2, figsize=(6, 3))
    for j, a in enumerate(ax2s):
        a.imshow(pca_dig.components_[j].reshape(8, 8), cmap='binary')
        a.set_title(f'PC{j+1}')
        a.axis('off')
    plt.suptitle('Главные компоненты как «шаблоны» пикселей')
    plt.tight_layout()

plt.tight_layout()

### Выбор числа компонент

Правило: сохраняем столько компонент, чтобы объяснить **не менее 90% дисперсии**.

In [ ]:
pca_full = PCA().fit(X_dig)
cumvar = np.cumsum(pca_full.explained_variance_ratio_)
n_90 = np.argmax(cumvar >= 0.90) + 1

plt.figure(figsize=(9, 4))
plt.plot(cumvar, color='steelblue', lw=2)
plt.axvline(n_90 - 1, color='orange', linestyle='--', label=f'{n_90} компонент')
plt.axhline(0.90,     color='red',    linestyle='--', label='90% дисперсии')
plt.xlabel('Число компонент')
plt.ylabel('Накопленная доля объяснённой дисперсии')
plt.title('Выбор числа компонент PCA (MNIST digits)')
plt.legend()
plt.grid(True, alpha=0.3)
print(f'Для объяснения 90% дисперсии нужно {n_90} компонент из {X_dig.shape[1]}')

### Сжатие как реконструкция

PCA не просто снижает размерность — он может **восстановить** приблизительный исходный объект через `inverse_transform`. Чем больше компонент, тем лучше качество реконструкции.

In [ ]:
sample = X_dig[42]  # одна цифра
n_components_list = [1, 2, 4, 8, 16, 32, 48, 64]

fig, axes = plt.subplots(1, len(n_components_list) + 1, figsize=(14, 2))
axes[0].imshow(sample.reshape(8, 8), cmap='binary')
axes[0].set_title('Исходная')
axes[0].axis('off')

for ax, n in zip(axes[1:], n_components_list):
    pca_n = PCA(n).fit(X_dig)
    recon = pca_n.inverse_transform(pca_n.transform(sample.reshape(1, -1)))
    ax.imshow(recon.reshape(8, 8), cmap='binary')
    ax.set_title(f'n={n}')
    ax.axis('off')
plt.suptitle('Реконструкция цифры при разном числе PCA-компонент', y=1.05)

### PCA vs t-SNE

PCA — линейный метод. Для нелинейных структур лучше работает **t-SNE**: он сохраняет локальные расстояния, но работает значительно медленнее и не строит явное преобразование (только визуализация).

In [ ]:
tsne = TSNE(random_state=17, n_iter=500)
X_dig_tsne = tsne.fit_transform(X_dig)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, X_2d, title in zip(
    axes,
    [X_dig_pca, X_dig_tsne],
    ['PCA (линейный, быстрый)', 't-SNE (нелинейный, медленный)']
):
    sc = ax.scatter(X_2d[:, 0], X_2d[:, 1], c=y_dig,
                    edgecolor='none', alpha=0.7, s=20,
                    cmap=plt.cm.get_cmap('tab10', 10))
    plt.colorbar(sc, ax=ax)
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
plt.suptitle('Визуализация MNIST digits: PCA vs t-SNE', y=1.02)
plt.tight_layout()

t-SNE разделяет кластеры значительно чище. Однако:
- t-SNE не даёт явного преобразования — нельзя применить к новым данным.
- Масштаб осей t-SNE не имеет смысла.
- PCA воспроизводим и интерпретируем.

---
## 2. PCA как предобработка

PCA часто используют перед классификацией: ускоряет обучение и снижает переобучение.

Пример: распознавание лиц (датасет LFW).

In [ ]:
import os
lfw_path = '../../data/faces'
os.makedirs(lfw_path, exist_ok=True)

try:
    lfw = datasets.fetch_lfw_people(min_faces_per_person=50, resize=0.4,
                                    data_home=lfw_path)
    print(f'{lfw.data.shape[0]} объектов, {lfw.data.shape[1]} признаков, {len(lfw.target_names)} классов')
    print('Персоны:', ', '.join(lfw.target_names))
except Exception as e:
    print('Датасет LFW недоступен (нет интернета или не загружен).')
    print('Используем датасет цифр как резервный пример.')
    lfw = None

In [ ]:
if lfw is not None:
    X_lfw_tr, X_lfw_te, y_lfw_tr, y_lfw_te = train_test_split(
        lfw.data, lfw.target, random_state=0
    )
    # PCA: 1850 признаков → 100
    pca_lfw = PCA(n_components=100, svd_solver='randomized')
    pca_lfw.fit(X_lfw_tr)
    var_exp = 100 * np.cumsum(pca_lfw.explained_variance_ratio_)[-1]
    print(f'100 компонент объясняют {var_exp:.1f}% дисперсии')

    # Классификация с PCA
    clf = LogisticRegression(multi_class='multinomial', solver='lbfgs',
                             random_state=17, max_iter=10000)
    clf.fit(pca_lfw.transform(X_lfw_tr), y_lfw_tr)
    acc_pca = accuracy_score(y_lfw_te, clf.predict(pca_lfw.transform(X_lfw_te)))

    # Без PCA
    clf2 = LogisticRegression(multi_class='multinomial', solver='lbfgs',
                              random_state=17, max_iter=10000)
    clf2.fit(X_lfw_tr, y_lfw_tr)
    acc_full = accuracy_score(y_lfw_te, clf2.predict(X_lfw_te))

    print(f'Точность с PCA (100 компонент): {acc_pca:.3f}')
    print(f'Точность без PCA (1850 признаков): {acc_full:.3f}')
else:
    # Резервный пример: MNIST digits
    X_tr, X_te, y_tr, y_te = train_test_split(X_dig, y_dig, random_state=0)
    pca_n = PCA(n_components=n_90)
    clf_pca = DecisionTreeClassifier(random_state=42)
    clf_pca.fit(pca_n.fit_transform(X_tr), y_tr)
    acc_pca = accuracy_score(y_te, clf_pca.predict(pca_n.transform(X_te)))

    clf_full = DecisionTreeClassifier(random_state=42)
    clf_full.fit(X_tr, y_tr)
    acc_full = accuracy_score(y_te, clf_full.predict(X_te))

    print(f'Дерево с PCA ({n_90} компонент): {acc_pca:.3f}')
    print(f'Дерево без PCA (64 признака):    {acc_full:.3f}')

---
## 3. Кластеризация: k-Means

### Задача кластеризации

Дано: набор объектов без меток. Нужно разбить их на группы так, чтобы объекты **внутри группы были похожи**, а **между группами — различались**.

### Алгоритм k-Means

Минимизирует суммарное квадратичное расстояние от объектов до центроидов своих кластеров:

$$J(C) = \sum_{k=1}^K \sum_{i \in C_k} \|x_i - \mu_k\|^2 \rightarrow \min_C$$

**Алгоритм (Lloyd's algorithm):**
1. Случайно разместить $K$ центроидов $\mu_1, \dots, \mu_K$
2. **E-шаг:** назначить каждый объект ближайшему центроиду
3. **M-шаг:** пересчитать центроиды как средние по кластерам
4. Повторять шаги 2–3 до сходимости

In [ ]:
# Ручная реализация для понимания
np.random.seed(42)
X_cl = np.zeros((150, 2))
X_cl[:50,   0] = np.random.normal(0.0, 0.3, 50);   X_cl[:50,   1] = np.random.normal(0.0, 0.3, 50)
X_cl[50:100,0] = np.random.normal(2.0, 0.5, 50);   X_cl[50:100,1] = np.random.normal(-1.0,0.2, 50)
X_cl[100:,  0] = np.random.normal(-1.0,0.2, 50);   X_cl[100:,  1] = np.random.normal(2.0, 0.5, 50)

# Запускаем итерации вручную
centroids = np.random.normal(0.0, 1.0, (3, 2))
hist = [centroids.copy()]
for _ in range(3):
    labels = cdist(X_cl, centroids).argmin(axis=1)
    centroids = np.array([X_cl[labels == k].mean(axis=0) for k in range(3)])
    hist.append(centroids.copy())

colors = ['steelblue', 'orange', 'green']
fig, axes = plt.subplots(1, 4, figsize=(15, 3))
for step, (ax, cents) in enumerate(zip(axes, hist)):
    dists  = cdist(X_cl, cents)
    labs   = dists.argmin(axis=1)
    for k, color in enumerate(colors):
        ax.scatter(X_cl[labs==k, 0], X_cl[labs==k, 1],
                   color=color, s=15, alpha=0.6)
        ax.scatter(cents[k, 0], cents[k, 1],
                   color=color, marker='X', s=200, edgecolors='black', zorder=5)
    ax.set_title(f'Итерация {step}')
    ax.grid(True, alpha=0.2)
plt.suptitle('k-Means: пошаговая визуализация', y=1.02)
plt.tight_layout()

**Важные особенности k-Means:**
- Чувствителен к начальным позициям центроидов → запускают несколько раз (`n_init`).
- Инициализация `k-means++` выбирает начальные центроиды максимально разнесёнными — работает значительно лучше случайной.
- Предполагает сферические кластеры одинакового размера — на вытянутых или неодинаковых кластерах работает плохо.
- Нужно масштабировать признаки (`StandardScaler`) перед кластеризацией.

### Пример: кластеризация рукописных цифр

In [ ]:
kmeans_dig = KMeans(n_clusters=10, random_state=42, n_init=10)
kmeans_dig.fit(X_dig)

ari = adjusted_rand_score(y_dig, kmeans_dig.labels_)
print(f'Adjusted Rand Index: {ari:.3f}  (1.0 = идеально, 0.0 = случайно)')

# Визуализируем центроиды кластеров
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for ax, center in zip(axes.ravel(), kmeans_dig.cluster_centers_):
    ax.imshow(center.reshape(8, 8), cmap='gray')
    ax.axis('off')
plt.suptitle('Центроиды кластеров k-Means на рукописных цифрах')
plt.tight_layout()

---
## 4. Выбор числа кластеров

Как определить оптимальное $K$? Один из методов — **метод локтя (Elbow method)**.

По мере роста $K$ инерция (суммарное расстояние до центроидов) убывает. «Локоть» — точка, где убывание резко замедляется.

In [ ]:
inertias = []
for k in range(1, 11):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_cl)
    inertias.append(km.inertia_)

plt.figure(figsize=(7, 4))
plt.plot(range(1, 11), inertias, marker='o', color='steelblue', lw=2)
plt.axvline(3, color='orange', linestyle='--', label='Оптимум K=3')
plt.xlabel('Число кластеров $K$')
plt.ylabel('Инерция $J(C)$')
plt.title('Метод локтя: выбор числа кластеров')
plt.legend()
plt.grid(True, alpha=0.3)

### Силуэтная метрика

**Силуэт** — более формальная метрика выбора $K$. Для каждого объекта $i$:

$$s(i) = \frac{b(i) - a(i)}{\max(a(i), b(i))}$$

где $a(i)$ — среднее расстояние до объектов **своего** кластера, $b(i)$ — среднее расстояние до объектов **ближайшего чужого** кластера.

$s(i) \in [-1, 1]$: чем ближе к 1 — тем лучше объект вписывается в свой кластер.

In [ ]:
sil_scores = []
for k in range(2, 11):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_cl)
    sil_scores.append(silhouette_score(X_cl, labels))

plt.figure(figsize=(7, 4))
plt.plot(range(2, 11), sil_scores, marker='o', color='orange', lw=2)
plt.xlabel('Число кластеров $K$')
plt.ylabel('Средний силуэт')
plt.title('Силуэтная метрика: максимум = оптимальное K')
plt.grid(True, alpha=0.3)
opt_k = range(2, 11)[np.argmax(sil_scores)]
plt.axvline(opt_k, color='steelblue', linestyle='--', label=f'Оптимум K={opt_k}')
plt.legend()
print(f'Оптимальное K по силуэту: {opt_k}')

---
## 5. Другие алгоритмы кластеризации

### Agglomerative Clustering (иерархическая кластеризация)

Не требует задавать $K$ заранее. Строит иерархию слияний:
1. Каждый объект — отдельный кластер.
2. Объединяем два ближайших кластера.
3. Повторяем до одного кластера.

Результат — **дендрограмма**: дерево слияний. Срезая её на нужной высоте, получаем нужное число кластеров.

In [ ]:
# Дендрограмма на небольшой выборке
np.random.seed(42)
X_small = X_cl[np.random.choice(150, 30, replace=False)]

dist_matrix = pdist(X_small)
Z = hierarchy.linkage(dist_matrix, method='ward')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

hierarchy.dendrogram(Z, ax=axes[0], color_threshold=2.0)
axes[0].set_title('Дендрограмма (иерархическая кластеризация)')
axes[0].set_xlabel('Объект')
axes[0].set_ylabel('Расстояние')
axes[0].grid(True, alpha=0.3)

# Сравнение с k-Means на тех же данных
agg = AgglomerativeClustering(n_clusters=3)
labels_agg = agg.fit_predict(X_small)
for k, color in enumerate(colors):
    axes[1].scatter(X_small[labels_agg==k, 0], X_small[labels_agg==k, 1],
                    color=color, s=60, edgecolors='white', label=f'Кластер {k+1}')
axes[1].set_title('AgglomerativeClustering (K=3)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.tight_layout()

### Сравнение алгоритмов на синтетических данных

Разные алгоритмы кластеризации имеют разные предположения о форме кластеров.

In [ ]:
from sklearn.datasets import make_moons, make_blobs, make_circles

np.random.seed(42)
datasets_cl = [
    (make_blobs(300, centers=3, random_state=0)[0],    'Сферические'),
    (make_moons(300, noise=0.08, random_state=0)[0],   'Полумесяцы'),
    (make_circles(300, noise=0.05, factor=0.5, random_state=0)[0], 'Кольца'),
]
algs = [
    ('k-Means',     KMeans(n_clusters=2, random_state=42, n_init=10)),
    ('Agglomerative', AgglomerativeClustering(n_clusters=2)),
    ('Spectral',    SpectralClustering(n_clusters=2, random_state=42,
                                       affinity='nearest_neighbors', n_neighbors=10)),
]

fig, axes = plt.subplots(len(algs), len(datasets_cl), figsize=(12, 9))
for col, (X_d, data_name) in enumerate(datasets_cl):
    Xs = StandardScaler().fit_transform(X_d)
    for row, (alg_name, alg) in enumerate(algs):
        labels = alg.fit_predict(Xs)
        for k in np.unique(labels):
            axes[row, col].scatter(Xs[labels==k, 0], Xs[labels==k, 1],
                                   s=15, alpha=0.7)
        if row == 0:
            axes[row, col].set_title(data_name)
        if col == 0:
            axes[row, col].set_ylabel(alg_name)
        axes[row, col].set_xticks([])
        axes[row, col].set_yticks([])
plt.suptitle('Алгоритмы кластеризации на разных типах данных', y=1.01)
plt.tight_layout()

**Выводы:**
- **k-Means**: хорош для выпуклых сферических кластеров, плохо справляется с кольцами и полумесяцами.
- **AgglomerativeClustering**: гибче, хорошо работает при малых данных и нужде в иерархии.
- **SpectralClustering**: справляется с нелинейными границами, но вычислительно дорог.

---
## 6. Метрики качества кластеризации

Оценить кластеризацию сложнее, чем классификацию: правильных меток обычно нет.

**Внешние метрики** (нужны истинные метки):

| Метрика | Что измеряет | Диапазон |
|---|---|---|
| Adjusted Rand Index (ARI) | Сходство разбиений | $[-1, 1]$, 1 = идеал |
| Homogeneity | Каждый кластер — один класс | $[0, 1]$ |
| Completeness | Каждый класс — один кластер | $[0, 1]$ |
| V-measure | Гармоническое среднее H и C | $[0, 1]$ |

**Внутренние метрики** (без истинных меток):

| Метрика | Что измеряет | Диапазон |
|---|---|---|
| Silhouette Score | Плотность внутри vs расстояние снаружи | $[-1, 1]$, 1 = идеал |

In [ ]:
# Сравнение алгоритмов на цифрах MNIST
X_d, y_d = load_digits().data, load_digits().target

algorithms_cmp = [
    ('k-Means',          KMeans(n_clusters=10, random_state=1, n_init=10)),
    ('Agglomerative',    AgglomerativeClustering(n_clusters=10)),
    ('Spectral',         SpectralClustering(n_clusters=10, random_state=1,
                                            affinity='nearest_neighbors')),
]

rows = []
for name, alg in algorithms_cmp:
    labels = alg.fit_predict(X_d)
    rows.append({
        'Алгоритм':    name,
        'ARI':         round(metrics.adjusted_rand_score(y_d, labels), 3),
        'Homogeneity': round(metrics.homogeneity_score(y_d, labels), 3),
        'Completeness':round(metrics.completeness_score(y_d, labels), 3),
        'V-measure':   round(metrics.v_measure_score(y_d, labels), 3),
        'Silhouette':  round(metrics.silhouette_score(X_d, labels, sample_size=500), 3),
    })

df_metrics = pd.DataFrame(rows).set_index('Алгоритм')
print(df_metrics.to_string())

### Практическое применение: поиск тематических кластеров в текстах

k-Means на TF-IDF векторах позволяет обнаружить латентные темы без разметки.

In [ ]:
from sklearn.datasets import fetch_20newsgroups

categories = ['sci.space', 'comp.graphics', 'alt.atheism', 'talk.religion.misc']
try:
    news = fetch_20newsgroups(subset='all', categories=categories,
                              shuffle=True, random_state=42)
    vect_news = TfidfVectorizer(max_df=0.5, max_features=1000,
                                min_df=2, stop_words='english')
    X_news = vect_news.fit_transform(news.data)
    y_news = news.target

    km_news = KMeans(n_clusters=4, init='k-means++', max_iter=100, n_init=5,
                     random_state=42)
    km_news.fit(X_news)

    print(f'ARI: {metrics.adjusted_rand_score(y_news, km_news.labels_):.3f}')
    print(f'V-measure: {metrics.v_measure_score(y_news, km_news.labels_):.3f}\n')

    terms = vect_news.get_feature_names_out()
    order = km_news.cluster_centers_.argsort()[:, ::-1]
    for i in range(4):
        top = [terms[j] for j in order[i, :10]]
        print(f'Кластер {i+1}: {" | ".join(top)}')
except Exception:
    print('Датасет 20newsgroups недоступен — пропускаем.')

---
## 7. Практика

Для заданий используем датасет рукописных цифр и датасет Iris.

In [ ]:
# Уже загружены выше
print(f'Digits: {X_dig.shape[0]} объектов, {X_dig.shape[1]} признаков, {len(np.unique(y_dig))} классов')
print(f'Iris:   {X_ir.shape[0]} объектов, {X_ir.shape[1]} признаков, {len(np.unique(y_ir))} класса')

### Задание 1
Примените PCA к датасету Iris (4 признака → 2 компоненты).

- Визуализируйте проекцию, раскрасив точки по классу.
- Выведите, какую долю дисперсии объясняет каждая компонента.
- Как называются классы? Какой из них хуже всего отделяется в 2D?

Затем обучите `DecisionTreeClassifier(max_depth=3)` на исходных признаках и на PCA-проекции. Сравните точность.

In [ ]:
# Ваш код здесь

### Задание 2
Постройте график накопленной объяснённой дисперсии PCA для датасета Digits.

- Сколько компонент нужно для 80%, 90%, 95% дисперсии?
- Обучите `KMeans(n_clusters=10)` на исходных данных и на PCA-сжатых (до 90% дисперсии). Сравните ARI.

In [ ]:
# Ваш код здесь

### Задание 3
Примените метод локтя и силуэтную метрику к датасету Iris для выбора числа кластеров k-Means.

- Переберите K от 2 до 8.
- Постройте оба графика.
- Какое K рекомендует каждый метод? Совпадает ли с числом классов (3)?

In [ ]:
# Ваш код здесь

### Задание 4
Сравните три алгоритма кластеризации на датасете Iris (K=3):
- `KMeans`
- `AgglomerativeClustering`
- `SpectralClustering`

Для каждого посчитайте ARI, V-measure и Silhouette Score. Представьте результаты в виде таблицы `pd.DataFrame`.

> Не забудьте масштабировать данные перед кластеризацией.

In [ ]:
# Ваш код здесь

### Задание 5 (повышенная сложность)
Полный пайплайн: PCA + кластеризация + визуализация.

1. Загрузите датасет Digits.
2. Сожмите до 2D с помощью PCA и до 2D с помощью t-SNE.
3. Примените `KMeans(n_clusters=10)` на оригинальных данных.
4. Визуализируйте результат кластеризации в PCA-пространстве и t-SNE-пространстве — раскрасьте точки по метке кластера и по истинному классу.
5. Постройте дендрограмму иерархической кластеризации на случайной подвыборке 50 объектов из Digits.

Какое пространство (PCA или t-SNE) лучше отражает качество кластеризации?

In [ ]:
# Ваш код здесь

---
## Полезные ресурсы

- [sklearn: Decomposition (PCA, SVD)](https://scikit-learn.org/stable/modules/decomposition.html)
- [sklearn: Clustering overview](https://scikit-learn.org/stable/modules/clustering.html)
- [sklearn: Clustering metrics](https://scikit-learn.org/stable/modules/clustering.html#clustering-performance-evaluation)
- [t-SNE: визуализация от автора](https://lvdmaaten.github.io/tsne/)
- [PCA Q&A на StackExchange](http://stats.stackexchange.com/questions/2691/making-sense-of-principal-component-analysis-eigenvectors-eigenvalues)